# 11_02 — Molecule-level calibration + aggregation (run AFTER the spectrum-model freeze)

Goal: replace Submission 1's reciprocal-rank fusion with a DEV-selected molecule aggregator.

* **Calibration.** `p = softmax(score / T)` per spectrum. T is fitted on the frozen model's
  **own training OOF predictions** (`oof_rows.parquet`, TL-like, never HOST), with log T in [-3, 3].
  NLL, Brier, ECE and reliability are diagnostics only.
* **Development molecules (MOL_DEV).** These are non-HOST, non-TL_EVAL connectivities with at
  least 3 spectra. They are disjoint from every training manifest, so fold-mean predictions are
  out-of-sample (asserted). There are 20 test-like multiplicity resamples: m spectra per
  molecule, m drawn from the visible test spectra-per-molecule distribution.
* **Aggregators.** A1 max score, A2 max prob, A3 mean prob (absent 0), A4 sum log prob (eps
  options fixed in advance), A5 log-mean-exp over present spectra, A6 RRF k=60, A7 most
  confident spectrum, A8 first spectrum.
* **Selection (DEV only, locked before HOST).** Take the best mean molecule MRR@25. Any
  aggregator whose paired molecule-bootstrap CI against the best includes 0 is within CI, and
  the simplest of those wins.
* **HOST.** Molecule-level confirmation only (molecule = connectivity).

In [ ]:
import sys, json
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists() and (REPO_ROOT.parent / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from casmi.paths import get_project_paths
from casmi.qcr.context import SEED, v4b_paths, v5_paths
from casmi.qcr.settlement import load_settlement_or_refuse
from casmi.ranking.calibration import calibration_report, fit_temperature
from casmi.ranking.lambdamart import load_fold_models
from casmi.ranking.manifests import load_manifest
from casmi.ranking.molecule_eval import molecule_rr, multiplicity_resamples, select_aggregator_dev_only
from casmi.ranking.rank_eval import rank_by_keys
from casmi.ranking.scaling import fold_mean_scores
from casmi.ranking.selection import DevSelectionLock, lock_dev_selection, require_dev_selection_lock, write_preregistration
from casmi.validation.cluster_bootstrap import cluster_bootstrap_mean, paired_cluster_bootstrap
from casmi.validation.decision_log import log_decision
from casmi_infer.aggregation import (FirstSpectrum, LogMeanExp, MaxProb, MaxScore, MeanProb, MostConfidentSpectrum, RRFAggregator,
                                     SumLogProb, softmax_by_group)

paths = get_project_paths()
vp, D = v4b_paths(paths), v5_paths(paths)
KEY = 'candidate_connectivity_key'
N_RESAMPLES, N_BOOT = 20, 2000
A4_EPS_OPTIONS = (1e-3, 1e-6, 1e-9)          # fixed BEFORE any result, DEV-only
PREREG, LOCK_PATH = D['freeze'] / 'preregistration_aggregator.json', D['freeze'] / 'aggregator_lock.json'

# GUARD (v5.1): refuse unless freeze_status == "FROZEN" exactly (PROVISIONAL refused), and re-verify the model-file
# sha256s, the feature-order hash and the frozen evidence-protocol hash BEFORE any aggregation work
from casmi.ranking.freeze import require_frozen_spectrum_model
freeze = require_frozen_spectrum_model(D['freeze'] / 'spectrum_model.json')
print('frozen spectrum model:', freeze['model_id'], '|', freeze['freeze_status'], '| verified hashes')
from casmi.qcr.protocols import PRIMARY_PROTOCOL
PROTOCOL = freeze.get('protocol')        # the frozen spectrum-evidence protocol -- every feature table below is built under it
if PROTOCOL != PRIMARY_PROTOCOL:
    raise RuntimeError(f'frozen spectrum model uses protocol {PROTOCOL!r}; v5.3 aggregation requires {PRIMARY_PROTOCOL!r} -- re-freeze with 10v5_02')
models, meta = load_fold_models(freeze['model_dir'])
FEATS = freeze['feature_names']
AGGS = [MaxScore(KEY), MaxProb(KEY), MeanProb(KEY), *[SumLogProb(eps=e, cand_col=KEY) for e in A4_EPS_OPTIONS], LogMeanExp(KEY),
        RRFAggregator(k=60, cand_col=KEY), MostConfidentSpectrum(KEY), FirstSpectrum(KEY)]
agg_id = lambda a: a.name if a.name != 'A4_sum_log_prob' else f'A4_sum_log_prob_eps{a.eps:g}'
SIMPLICITY = {agg_id(a): a.simplicity for a in AGGS}
prereg = write_preregistration(PREREG, rule={
    'rule_id': 'v5-aggregator-1', 'population': 'MOL_DEV (non-HOST, non-TL_EVAL, training-disjoint; >=3 spectra/connectivity)',
    'metric': 'molecule MRR@25 averaged over 20 test-like multiplicity resamples', 'selection': 'best mean; within paired molecule-bootstrap CI of best -> simplest',
    'aggregators': list(SIMPLICITY), 'simplicity': SIMPLICITY, 'temperature': 'fit on frozen model training OOF only; log T in [-3,3]',
    'host_role': 'confirmation only', 'tie_rule': ['aggregator score DESC', 'mean abs ppm ASC', 'connectivity_key ASC']})
print('aggregator rule sha256:', prereg['rule_sha256'])

## 1. Temperature from the frozen model's own training OOF (TL-like; never HOST)

In [ ]:
oof = pd.read_parquet(Path(freeze['model_dir']) / 'oof_rows.parquet')
temp = fit_temperature(oof, score_col='score', group_col='query_id', truth_col='is_true_candidate')
T = temp['temperature']
print(json.dumps(temp, indent=1))
cal_oof, rel_oof = calibration_report(oof, T)
print('calibration on training OOF (diagnostic):', cal_oof)

## 2. MOL_DEV spectrum predictions (out-of-sample fold-mean) + calibration diagnostics

In [ ]:
mol_man, mol_meta = load_manifest('MOL_DEV', D['manifests'])
train_man, _ = load_manifest(freeze['manifest'], D['manifests'])
assert not (set(mol_man['connectivity_key']) & set(train_man['connectivity_key'])), 'MOL_DEV overlaps the frozen model training structures'
from casmi.qcr.protocol_features import artifact_status, build_host_protocol_artifacts, build_protocol_artifacts, feature_path, qcr_status
settlement = load_settlement_or_refuse(vp.settlement_json)
if artifact_status(D, 'MOL_DEV', PROTOCOL, settlement['evidence_fingerprint']) != 'READY':
    if qcr_status(D, 'MOL_DEV') != 'READY':
        raise RuntimeError('MOL_DEV QCR missing. Run: python scripts/v5_build_scale_features.py --manifest MOL_DEV '
                           '&& python scripts/v5_build_protocol_features.py --manifest MOL_DEV')
    print(build_protocol_artifacts(D, 'MOL_DEV', PROTOCOL, settlement['evidence_fingerprint'], derive_training_manifest=False))
mf = pd.read_parquet(feature_path(D, 'MOL_DEV', PROTOCOL))
mf = mf.assign(score=fold_mean_scores(models, mf, FEATS))
mf = rank_by_keys(mf, [('score', False), ('abs_mass_error_ppm', True), (KEY, True)])
per_spec = mf.rename(columns={'query_id': 'spectrum_id'}).assign(molecule_id=lambda d: d['true_connectivity_key'])
per_spec['prob'] = softmax_by_group(per_spec['score'], per_spec['spectrum_id'], T)
truth_of_mol = {m: m for m in mol_man['connectivity_key'].unique()}   # a development molecule IS its connectivity
cal_dev, rel_dev = calibration_report(mf, T)
print('calibration on MOL_DEV spectra (diagnostic):', cal_dev)
display(rel_dev)
fig, ax = plt.subplots(figsize=(4.5, 4))
for rel, lab in ((rel_oof, 'training OOF'), (rel_dev, 'MOL_DEV')):
    r = rel[rel['n'] > 0]
    ax.plot(r['mean_confidence'], r['accuracy'], 'o-', label=lab)
ax.plot([0, 1], [0, 1], 'k:'); ax.set_xlabel('top-1 confidence'); ax.set_ylabel('top-1 accuracy'); ax.legend(); ax.set_title(f'Reliability (T={T:.3g})')
fig.tight_layout(); fig.savefig(D['figures'] / 'reliability_temperature.png', dpi=130); plt.show()

## 3. Aggregators under 20 test-like multiplicity resamples (DEV only)

In [ ]:
test_map = pd.read_parquet(paths.processed / 'test_molecule_map.parquet')
test_mult = test_map['n_spectra'].to_numpy()
rr_rows = []
for r, sub in multiplicity_resamples(per_spec, test_mult, n_resamples=N_RESAMPLES, seed=SEED):
    for a in AGGS:
        rr = molecule_rr(a.rank(sub), truth_of_mol, KEY)
        rr_rows.append(pd.DataFrame({'resample': r, 'aggregator': agg_id(a), 'molecule_id': rr.index, 'rr': rr.to_numpy()}))
rr_long = pd.concat(rr_rows, ignore_index=True)
rr_long.to_parquet(D['metrics'] / 'molecule_dev_rr_long.parquet', index=False)
per_mol = rr_long.groupby(['aggregator', 'molecule_id'])['rr'].mean().unstack(0)          # molecule x aggregator (mean over resamples)
by_resample = rr_long.groupby(['aggregator', 'resample'])['rr'].mean().unstack(0)
summary = pd.DataFrame({'dev_mrr_mean': by_resample.mean(), 'dev_mrr_sd_over_resamples': by_resample.std(ddof=1)})
best = summary['dev_mrr_mean'].idxmax()
mols = per_mol.index.to_numpy()
rows = []
for a in per_mol.columns:
    ci = cluster_bootstrap_mean(per_mol[a].to_numpy(), mols, n_boot=N_BOOT, seed=SEED)
    d = paired_cluster_bootstrap(per_mol[a].to_numpy(), per_mol[best].to_numpy(), mols, n_boot=N_BOOT, seed=SEED)
    rows.append({'aggregator': a, 'ci_low': ci['ci_low'], 'ci_high': ci['ci_high'], 'delta_vs_best': d['delta'],
                 'delta_ci_low': d['ci_low'], 'delta_ci_high': d['ci_high']})
summary = summary.reset_index().rename(columns={'index': 'aggregator'}).merge(pd.DataFrame(rows), on='aggregator')
display(summary.sort_values('dev_mrr_mean', ascending=False))

## 4. Pre-registered DEV selection → LOCK (before HOST)

In [ ]:
SELECTED, best_raw, ranked = select_aggregator_dev_only(summary[['aggregator', 'dev_mrr_mean', 'delta_ci_low', 'delta_ci_high']], SIMPLICITY)
sel_obj = next(a for a in AGGS if agg_id(a) == SELECTED)
agg_cfg = {**sel_obj.config(), 'temperature': T if sel_obj.needs_prob else None, 'aggregator_id': SELECTED}
lock_dev_selection(LOCK_PATH, SELECTED, ranked, {'temperature': T}, PREREG, extra={'aggregator_config': agg_cfg, 'best_raw': best_raw})
log_decision(decision=f'v5 molecule aggregator (MOL_DEV rule): {SELECTED}', evidence_used='molecule_dev_rr_long.parquet; no HOST metric yet',
             host_visible=False, category='pre_registered_choice', log_path=paths.outputs / 'decision_log.jsonl')
display(ranked)
print('SELECTED aggregator:', SELECTED, agg_cfg)

## 4b. MOL_DEV molecule metrics by spectrum-regime mix (REPORT ONLY; the selection above is locked)

In [ ]:
from casmi.validation.regime_audit import load_regimes, molecule_regime_mix
REG_DIR = D['root'] / 'regimes'
MIX_GROUPS = ('ALL_MODE_A', 'MIXED', 'ALL_REF_ABSENT', 'OTHER')


def by_mix(per_molecule_rr, mix):
    d = per_molecule_rr.join(mix.set_index('connectivity_key')['molecule_regime_mix'], how='left')
    rows = [{'group': 'ALL', 'n_molecules': len(d), **d.drop(columns='molecule_regime_mix').mean().to_dict()}]
    for g in MIX_GROUPS:
        sub = d[d['molecule_regime_mix'] == g]
        rows.append({'group': g, 'n_molecules': len(sub), **sub.drop(columns='molecule_regime_mix').mean().to_dict()})
    return pd.DataFrame(rows)


if (REG_DIR / 'MOL_DEV_query_regimes.parquet').exists():
    MOL_MIX = molecule_regime_mix(load_regimes('MOL_DEV', REG_DIR))
    display(by_mix(per_mol, MOL_MIX).round(4))
else:
    print('MOL_DEV regimes not built -- run 10v5_03 after the MOL_DEV feature build')

## 5. HOST molecule-level confirmation (molecule = connectivity; descriptive only)

In [ ]:
LOCK = require_dev_selection_lock(LOCK_PATH, PREREG)
settlement = load_settlement_or_refuse(vp.settlement_json)
host_q = pd.read_parquet(vp.host_processed / 'host_holdout_spectra.parquet')
if artifact_status(D, 'HOST', PROTOCOL, settlement['evidence_fingerprint']) != 'READY':
    build_host_protocol_artifacts(D, settlement['evidence_artifacts'], host_q, PROTOCOL, settlement['evidence_fingerprint'])
hf = pd.read_parquet(feature_path(D, 'HOST', PROTOCOL))
hf = rank_by_keys(hf.assign(score=fold_mean_scores(models, hf, FEATS)), [('score', False), ('abs_mass_error_ppm', True), (KEY, True)])
host_spec = hf.rename(columns={'query_id': 'spectrum_id'}).assign(molecule_id=lambda d: d['true_connectivity_key'])
host_spec['prob'] = softmax_by_group(host_spec['score'], host_spec['spectrum_id'], T)
host_truth = {m: m for m in host_q['true_connectivity_key'].unique()}
host_rows = []
for a in [sel_obj, RRFAggregator(k=60, cand_col=KEY), FirstSpectrum(KEY)]:
    rr_all = molecule_rr(a.rank(host_spec), host_truth, KEY)
    ci = cluster_bootstrap_mean(rr_all.to_numpy(), rr_all.index.to_numpy(), n_boot=N_BOOT, seed=SEED)
    rs = [molecule_rr(a.rank(sub), host_truth, KEY).mean() for _, sub in multiplicity_resamples(host_spec, test_mult, N_RESAMPLES, SEED)]
    host_rows.append({'aggregator': agg_id(a), 'host_mrr_all_spectra': ci['mean'], 'ci_low': ci['ci_low'], 'ci_high': ci['ci_high'],
                      'host_mrr_testlike_mean': float(np.mean(rs)), 'host_mrr_testlike_sd': float(np.std(rs, ddof=1)), 'n_molecules': ci['n_queries']})
host_tab = pd.DataFrame(host_rows)
display(host_tab)
if (REG_DIR / 'HOST_query_regimes.parquet').exists():
    HOST_MIX = molecule_regime_mix(load_regimes('HOST', REG_DIR))
    host_rr = pd.DataFrame({agg_id(a): molecule_rr(a.rank(host_spec), host_truth, KEY) for a in [sel_obj, RRFAggregator(k=60, cand_col=KEY)]})
    print('HOST molecule MRR@25 by spectrum-regime mix (confirmation only):')
    display(by_mix(host_rr, HOST_MIX).round(4))

## 6. Export the aggregator config (the bundle swap is a config change)

In [ ]:
out = {'aggregator': agg_cfg, 'temperature_fit': temp, 'calibration_training_oof': cal_oof, 'calibration_mol_dev': cal_dev,
       'dev_summary': summary.to_dict('records'), 'host_confirmation': host_tab.to_dict('records'), 'spectrum_model': freeze['model_id'],
       'rule_sha256': prereg['rule_sha256']}
(D['freeze'] / 'aggregator.json').write_text(json.dumps(out, indent=2, default=str), encoding='utf-8')
UPDATE_BUNDLE = False   # True: write this aggregator into bundle/config.json (new CONFIG_HASH) and refresh the manifest
if UPDATE_BUNDLE:
    from casmi import bundle_export as bx
    cfg = json.loads((D['bundle'] / 'config.json').read_text(encoding='utf-8'))
    new_cfg = bx.build_config(settlement['similarity_config'], aggregator=agg_cfg, top_k=cfg['top_k_submission'])
    (D['bundle'] / 'config.json').write_text(json.dumps(new_cfg, indent=2), encoding='utf-8')
    m = bx.refresh_manifest(D['bundle'])
    print('bundle updated: CONFIG_HASH', new_cfg['CONFIG_HASH'])